In [127]:
import os
import xgboost as xgb
import pickle as pkl
import pandas as pd
from tqdm import tqdm
import random as random
import numpy as np
from ray import tune

from custom_models.custom_model import BatchVectorizerXGBoost

from sklearn.metrics import mean_absolute_error, log_loss, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import mean_squared_error as mse
from sklearn.utils.multiclass import unique_labels
from sklearn.feature_extraction.text import HashingVectorizer

# read in data
# dtrain = xgb.DMatrix('agaricus.txt.train')
# dtest = xgb.DMatrix('agaricus.txt.test')
# # specify parameters via map
# param = {'max_depth':2, 'eta':1, 'objective':'binary:logistic'}
# num_round = 2
# bst = xgb.train(param, dtrain, num_round)
# # make prediction
# preds = bst.predict(dtest)

In [128]:
    neg_to_pos_ratio = 1

    results_file = "results_titles_abstracts_gbdt.json"

    inputs = "titles_abstracts"

    filename_pos_human = "../data/pubmed_articles_pos_human.csv"
    filename_neg_human = "../data/pubmed_articles_neg_human.csv"

    with open("../data/cleaned_pos_neg_train_val_test_dataset_ids.pkl", "rb") as f:
        prop_typ_id_dict = pkl.load(f)

    # ptypes = prop_typ_id_dict.keys()
#     ptypes = ["is_cohort", "is_guideline", "is_consensus", "is_qual", "is_ctprotocol"]
    ptype = 'is_consensus'


In [165]:
ptypes = prop_typ_id_dict.keys()

In [166]:
ptypes

dict_keys(['is_rct', 'is_sr', 'is_casecontrol', 'is_ct', 'is_cohort', 'is_guideline', 'is_consensus', 'is_qual', 'is_ctprotocol'])

In [129]:
            pos_data_human = pd.read_csv(filename_pos_human, chunksize=100000)
            neg_data_human = pd.read_csv(filename_neg_human, chunksize=100000)

            pos_train_ids = prop_typ_id_dict[ptype]["pos"]["train"]
            neg_train_ids = prop_typ_id_dict[ptype]["neg"]["train"]

            pos_val_ids = prop_typ_id_dict[ptype]["pos"]["val"]
            neg_val_ids = prop_typ_id_dict[ptype]["neg"]["val"]

            pos_test_ids = prop_typ_id_dict[ptype]["pos"]["test"]
            neg_test_ids = prop_typ_id_dict[ptype]["neg"]["test"]

            pos_train_set = pd.DataFrame()
            pos_val_set = pd.DataFrame()
            pos_test_set = pd.DataFrame()

            for pos_chunk in tqdm(pos_data_human):

                indices = pos_chunk.index
                indices = list(indices)

                pos_train_indices = list(
                    set(indices).intersection(set(pos_train_ids))
                )
                pos_val_indices = list(set(indices).intersection(set(pos_val_ids)))
                pos_test_indices = list(set(indices).intersection(set(pos_test_ids)))

                pos_train_subset = pos_chunk.loc[pos_train_indices]
                if pos_train_set.empty:
                    pos_train_set = pd.DataFrame(columns=pos_train_set.columns)
                pos_train_set = pd.concat([pos_train_set, pos_train_subset])

                pos_val_subset = pos_chunk.loc[pos_val_indices]
                if pos_val_set.empty:
                    pos_val_set = pd.DataFrame(columns=pos_val_set.columns)
                pos_val_set = pd.concat([pos_val_set, pos_val_subset])

                pos_test_subset = pos_chunk.loc[pos_test_indices]
                if pos_test_set.empty:
                    pos_test_set = pd.DataFrame(columns=pos_test_set.columns)
                pos_test_set = pd.concat([pos_test_set, pos_test_subset])

            pos_train_set[ptype] = int(1)
            pos_val_set[ptype] = int(1)
            pos_test_set[ptype] = int(1)

            del pos_data_human
            del pos_train_subset
            del pos_chunk
            del indices
            del pos_train_ids

            neg_train_ids_sample = random.sample(
                neg_train_ids, round(len(pos_train_set) * neg_to_pos_ratio)
            )

            neg_train_set = pd.DataFrame()
            neg_val_set = pd.DataFrame()
            neg_test_set = pd.DataFrame()

            for neg_chunk in tqdm(neg_data_human):

                indices = neg_chunk.index
                indices = list(indices)
                neg_train_indices = list(
                    set(indices).intersection(set(neg_train_ids_sample))
                )
                neg_val_indices = list(set(indices).intersection(set(neg_val_ids)))
                neg_test_indices = list(set(indices).intersection(set(neg_test_ids)))
                neg_subset = neg_chunk.loc[indices]

                neg_train_subset = neg_chunk.loc[neg_train_indices]
                if neg_train_set.empty:
                    neg_train_set = pd.DataFrame(columns=neg_train_set.columns)
                neg_train_set = pd.concat([neg_train_set, neg_train_subset])

                neg_val_subset = neg_chunk.loc[neg_val_indices]
                if neg_val_set.empty:
                    neg_val_set = pd.DataFrame(columns=neg_val_set.columns)
                neg_val_set = pd.concat([neg_val_set, neg_val_subset])

                neg_test_subset = neg_chunk.loc[neg_test_indices]
                if neg_test_set.empty:
                    neg_test_set = pd.DataFrame(columns=neg_test_set.columns)
                neg_test_set = pd.concat([neg_test_set, neg_test_subset])

            neg_train_set[ptype] = int(0)
            neg_val_set[ptype] = int(0)
            neg_test_set[ptype] = int(0)

            del neg_data_human
            del neg_subset
            del neg_chunk
            del indices

            train_data = pd.concat([neg_train_set, pos_train_set]).sample(
                frac=1
            )
            del neg_train_set
            del pos_train_set
            val_data = pd.concat([neg_val_set, pos_val_set]).sample(frac=1)
            del neg_val_set
            del pos_val_set
            test_data = pd.concat([neg_test_set, pos_test_set]).sample(frac=1)
            del neg_test_set
            del pos_test_set

            y_train = train_data[ptype]
            y_val = val_data[ptype]
            y_test = test_data[ptype]
            if inputs == "abstracts":
                X_train = train_data["abstract_plaintext"]
                X_val = val_data["abstract_plaintext"]
                X_test = test_data["abstract_plaintext"]
            elif inputs == "titles":
                X_train = train_data["title"]
                X_val = val_data["title"]
                X_test = test_data["title"]
            elif inputs == "titles_abstracts":
                X_train = pd.Series(
                    [
                        "{0} {1}".format(train_title, train_abstract)
                        for train_title, train_abstract in zip(
                            train_data["title"], train_data["abstract_plaintext"]
                        )
                    ]
                )
                X_val = pd.Series(
                    [
                        "{0} {1}".format(val_title, val_abstract)
                        for val_title, val_abstract in zip(
                            val_data["title"], val_data["abstract_plaintext"]
                        )
                    ]
                )
                X_test = pd.Series(
                    [
                        "{0} {1}".format(test_title, test_abstract)
                        for test_title, test_abstract in zip(
                            test_data["title"], test_data["abstract_plaintext"]
                        )
                    ]
                )

            del train_data
            del val_data
            del test_data

19it [00:27,  1.40s/it]/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/tqdm/std.py:1195: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
51it [01:11,  1.37s/it]/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/tqdm/std.py:1195: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
86it [01:58,  1.38s/it]
57it [01:38,  1.72s/it]


In [ ]:
batch_size = 100000
vectorizer = HashingVectorizer(ngram_range=(1, 4), stop_words="english")
params = {'objective': log_loss, 'verbose': False}

In [ ]:

vec_fitted = False

        # number of batches to be processed
if len(y_train) >= batch_size:
    num_batches = len(y) // batch_size
else:
    num_batches = 1

        # Store the classes seen during fit
#         self.classes_ = unique_labels(y)
for X_chunk, y_chunk in zip(
            np.array_split(X_train, num_batches), np.array_split(y_train, num_batches)
        ):

            # convert text chunk to unicode
            X_chunk = X_chunk.values.astype("U")

            if not vec_fitted:
                X_trans = vectorizer.fit_transform(X_chunk)
                vec_fitted = True
            else:
                X_trans = vectorizer.transform(X_chunk)
            del X_chunk
            
            xg_chunk = xgb.DMatrix(X_trans, label=y_chunk)
            if os.path.exists('model_1.model'):
                model_1 = xgb.train(params, xg_chunk, 30, xgb_model='model_1.model')
            else:
                model_1 = xgb.train(params, xg_chunk, 30)
            model_1.save_model('model_1.model')

In [ ]:
X_test_trans = X_test.values.astype("U")
X_test_trans = vectorizer.transform(X_test_trans)
xg_test = xgb.DMatrix(X_test_trans, label=y_test)

In [ ]:
predictions = np.array(list(map(int, np.round(model_1.predict(xg_test)))))

In [ ]:
set(predictions)

In [ ]:
print(precision_score(predictions, y_test))  

In [ ]:
# from sklearn.datasets import load_diabetes, load_iris
# X, y = load_diabetes(return_X_y=True)
# X, y = load_iris(return_X_y=True)

In [ ]:
# reg = xgb.XGBRegressor(
#     tree_method="hist",
#     eval_metric=mean_absolute_error,
# )
# reg.fit(X, y, eval_set=[(X, y)])

In [119]:

# clas = BatchVectorizerXGBoost(
# #     tree_method="hist",
#     objective="binary:hinge",
#     max_depth = 3,
#     gamma = 2,
#     reg_alpha = 50,
#     reg_lambda = 0.3,
#     colsample_bytree = 0.6,
#     min_child_weight = 4,
#     n_estimators = 180,
#     seed = 0)

# X_train_ = np.array(X_train)
# X_val_ = np.array(X_val)
# y_train_ = np.array(y_train)
# y_val_ = np.array(y_val)

# clas.fit(X_train, y_train, eval_set=[(X_val, y_val)])

In [124]:
# batch_size = 10
# clas = xgb.XGBClassifier(
    
#     tree_method="hist",
#     objective="binary:hinge",
#     max_depth = 3,
#     gamma = 2,
#     reg_alpha = 50,
#     reg_lambda = 0.3,
#     colsample_bytree = 0.6,
#     min_child_weight = 4,
#     n_estimators = 180,
#     seed = 0)
    

# vec_fitted = False
# model_fitted = False

#         # number of batches to be processed
# if len(y_train) >= batch_size:
#     num_batches = len(y_train) // batch_size
# else:
#     num_batches = 1

#         # Store the classes seen during fit
# #         self.classes_ = unique_labels(y)
# for X_chunk, y_chunk in zip(
#             np.array_split(X_train, num_batches), np.array_split(y_train, num_batches)
#         ):

#             # convert text chunk to unicode
#             X_chunk = X_chunk.values.astype("U")

#             if not vec_fitted:
#                 X_trans = vectorizer.fit_transform(X_chunk)
#                 vec_fitted = True
#             else:
#                 X_trans = vectorizer.transform(X_chunk)
#             del X_chunk
            
#             xg_chunk = xgb.DMatrix(X_trans, label=y_chunk)
#             y_chunk = np.array(y_chunk).reshape(-1)
    
#             if model_fitted:
#                 clas.fit(X_trans, y_chunk, xgb_model = clas.get_booster())
#                 print("fitted model")
#             else:
#                 clas.fit(X_trans, y_chunk)
#                 model_fitted = True
#                 print("fitting model")
# #                      eval_set=[(X_val, y_val)])
# #             if os.path.exists('model_1.model'):
# #                 model_1 = xgb.train(params, xg_chunk, 30, xgb_model='model_1.model')
# #             else:
# #                 model_1 = xgb.train(params, xg_chunk, 30)
#             model_1.save_model('model_1.model')

In [138]:
config = {
            "objective": tune.grid_search(
                [
                    "binary:hinge",
                    "rank:pairwise",
                    "rank:ndcg",
                    "rank:map",
                    "reg:squarederror",
                    "reg:squaredlogerror",
                    "reg:logistic",
                    "reg:pseudohubererror",
                    "binary:logistic",
                    "binary:logitraw",
                    "reg:linear",
                    "count:poisson",
                    "survival:cox",
                    "reg:tweedie",
                ]
            ),
            "max_depth": tune.grid_search(list(np.linspace(3, 18, 16).astype(int))),
            "gamma": tune.grid_search(list(np.linspace(0, 10, 10))),
            "reg_alpha": tune.grid_search(list(np.logspace(-5, 5, 10))),
            "reg_lambda": tune.grid_search(list(np.logspace(-5, 5, 10))),
            "colsample_bytree": tune.grid_search(list(np.linspace(0.5, 1, 5))),
            "min_child_weight": tune.grid_search(list(np.linspace(0, 10, 10))),
            "eta": tune.grid_search(list(np.logspace(-5, 0, 10))),
            "n_estimators": tune.grid_search([50, 65, 80, 100, 115, 130, 150, 180]),
            "seed": tune.grid_search([0]),
        }

In [139]:
def objective(config, data):
            bvc = BatchVectorizerXGBoost(
                objective=config["objective"],
                max_depth=config["max_depth"],
                gamma=config["gamma"],
                reg_alpha=config["reg_alpha"],
                reg_lambda=config["reg_lambda"],
                colsample_bytree=config["colsample_bytree"],
                min_child_weight=config["min_child_weight"],
                n_estimators=config["n_estimators"],
                seed=config["seed"],
            )
            bvc.fit(data["X_train"], data["y_train"])
            y_pred_val = bvc.predict(data["X_val"])
            f1 = metrics.f1_score(y_pred_val, data["y_val"])

            tune.report(f1=f1)

            return {"f1": f1}

       
        

In [140]:
len(data["y_train"])

9920

In [141]:
data = {
            "X_train": X_train,
            "y_train": y_train,
            "X_val": X_val,
            "y_val": y_val,
        }
analysis = tune.run(
            tune.with_parameters(objective, data=data),
            config=config,
            resources_per_trial={"gpu": 1},
            verbose=3,
        )



/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/ray/tune/suggest/basic_variant.py:324: UserWarning: The number of pre-generated samples (1088000000) exceeds the serialization threshold (1000000). Resume ability is disabled. To fix this, reduce the number of dimensions/size of the provided grid search.
  warnings.warn(
2022-07-28 14:29:45,394	INFO trial_runner.py:803 -- starting objective_5838c_00000
(pid=717192) /home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
(pid=717192)   from pandas import MultiIndex, Int64Index


Trial name,status,loc,colsample_bytree,eta,gamma,max_depth,min_child_weight,n_estimators,objective,reg_alpha,reg_lambda,seed
objective_5838c_00000,RUNNING,10.151.122.12:717192,0.5,1e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00001,PENDING,,0.625,1e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00002,PENDING,,0.75,1e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00003,PENDING,,0.875,1e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00004,PENDING,,1,1e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00005,PENDING,,0.5,3.59381e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00006,PENDING,,0.625,3.59381e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00007,PENDING,,0.75,3.59381e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00008,PENDING,,0.875,3.59381e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00009,PENDING,,1,3.59381e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0


(objective pid=717192) /home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/xgboost/sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
(objective pid=717192)   warnings.warn(label_encoder_deprecation_msg, UserWarning)


Trial name,status,loc,colsample_bytree,eta,gamma,max_depth,min_child_weight,n_estimators,objective,reg_alpha,reg_lambda,seed
objective_5838c_00000,RUNNING,10.151.122.12:717192,0.5,1e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00001,PENDING,,0.625,1e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00002,PENDING,,0.75,1e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00003,PENDING,,0.875,1e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00004,PENDING,,1,1e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00005,PENDING,,0.5,3.59381e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00006,PENDING,,0.625,3.59381e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00007,PENDING,,0.75,3.59381e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00008,PENDING,,0.875,3.59381e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00009,PENDING,,1,3.59381e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0


Trial name,status,loc,colsample_bytree,eta,gamma,max_depth,min_child_weight,n_estimators,objective,reg_alpha,reg_lambda,seed
objective_5838c_00000,RUNNING,10.151.122.12:717192,0.5,1e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00001,PENDING,,0.625,1e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00002,PENDING,,0.75,1e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00003,PENDING,,0.875,1e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00004,PENDING,,1,1e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00005,PENDING,,0.5,3.59381e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00006,PENDING,,0.625,3.59381e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00007,PENDING,,0.75,3.59381e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00008,PENDING,,0.875,3.59381e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00009,PENDING,,1,3.59381e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0


2022-07-28 14:30:00,380	WARNING tune.py:650 -- SIGINT received (e.g. via Ctrl+C), ending Ray Tune run. This will try to checkpoint the experiment state one last time. Press CTRL+C one more time (or send SIGINT/SIGKILL/SIGTERM) to skip. 


Trial name,status,loc,colsample_bytree,eta,gamma,max_depth,min_child_weight,n_estimators,objective,reg_alpha,reg_lambda,seed
objective_5838c_00000,RUNNING,10.151.122.12:717192,0.5,1e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00001,PENDING,,0.625,1e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00002,PENDING,,0.75,1e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00003,PENDING,,0.875,1e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00004,PENDING,,1,1e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00005,PENDING,,0.5,3.59381e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00006,PENDING,,0.625,3.59381e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00007,PENDING,,0.75,3.59381e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00008,PENDING,,0.875,3.59381e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00009,PENDING,,1,3.59381e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0


Trial name,status,loc,colsample_bytree,eta,gamma,max_depth,min_child_weight,n_estimators,objective,reg_alpha,reg_lambda,seed
objective_5838c_00000,RUNNING,10.151.122.12:717192,0.5,1e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00001,PENDING,,0.625,1e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00002,PENDING,,0.75,1e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00003,PENDING,,0.875,1e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00004,PENDING,,1,1e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00005,PENDING,,0.5,3.59381e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00006,PENDING,,0.625,3.59381e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00007,PENDING,,0.75,3.59381e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00008,PENDING,,0.875,3.59381e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0
objective_5838c_00009,PENDING,,1,3.59381e-05,0,3,0,50,binary:hinge,1e-05,1e-05,0


(objective pid=717192) 2022-07-28 14:30:44,620	ERROR function_runner.py:281 -- Runner Thread raised error.
(objective pid=717192) Traceback (most recent call last):
(objective pid=717192)   File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/ray/tune/function_runner.py", line 272, in run
(objective pid=717192)     self._entrypoint()
(objective pid=717192)   File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/ray/tune/function_runner.py", line 348, in entrypoint
(objective pid=717192)     return self._trainable_func(
(objective pid=717192)   File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/ray/util/tracing/tracing_helper.py", line 462, in _resume_span
(objective pid=717192)     return method(self, *_args, **_kwargs)
(objective pid=717192)   File "/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/ray/tune/function_runner.py", line 640, in _trainab

In [ ]:
best_config = analysis.get_best_config(metric="f1", mode="max")
        bvc = BatchVectorizerXGBoost()

In [161]:
clas = BatchVectorizerXGBoost(
    objective="reg:tweedie",
    max_depth = 3,
    gamma = 2,
    reg_alpha = 50,
    reg_lambda = 0.3,
    colsample_bytree = 0.6,
    min_child_weight = 4,
    n_estimators = 180,
    seed = 0)

clas.fit(data['X_train'], data['y_train'])

In [153]:
# clas

In [148]:
y_pred = clas.predict(X_val)
accuracy_score(y_pred, y_val)

AttributeError: 'BatchVectorizerXGBoost' object has no attribute 'model'

In [96]:
list(np.logspace(-5, 0, 10))

[1e-05,
 3.5938136638046256e-05,
 0.0001291549665014884,
 0.0004641588833612782,
 0.0016681005372000592,
 0.005994842503189409,
 0.021544346900318846,
 0.07742636826811278,
 0.2782559402207126,
 1.0]

In [92]:
np.linspace(3, 18, 1).astype(int)

array([3])

In [103]:
np.linspace(3, 18, 16).astype(int)

array([ 3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18])

In [111]:
list(np.linspace(0.5, 1, 5))

[0.5, 0.625, 0.75, 0.875, 1.0]

In [164]:
help(tune.run)

Help on function run in module ray.tune.tune:

run(run_or_experiment: Union[str, Callable, Type], name: Union[str, NoneType] = None, metric: Union[str, NoneType] = None, mode: Union[str, NoneType] = None, stop: Union[NoneType, Mapping, ray.tune.stopper.Stopper, Callable[[str, Mapping], bool]] = None, time_budget_s: Union[NoneType, int, float, datetime.timedelta] = None, config: Union[Dict[str, Any], NoneType] = None, resources_per_trial: Union[NoneType, Mapping[str, Union[float, int, Mapping]], ray.tune.utils.placement_groups.PlacementGroupFactory] = None, num_samples: int = 1, local_dir: Union[str, NoneType] = None, search_alg: Union[ray.tune.suggest.suggestion.Searcher, ray.tune.suggest.search.SearchAlgorithm, str, NoneType] = None, scheduler: Union[ray.tune.schedulers.trial_scheduler.TrialScheduler, str, NoneType] = None, keep_checkpoints_num: Union[int, NoneType] = None, checkpoint_score_attr: Union[str, NoneType] = None, checkpoint_freq: int = 0, checkpoint_at_end: bool = False, v